In [ ]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("./data/dataset")   # change if data lives elsewhere
OUTPUTS_DIR  = Path("./outputs")

MODELS_DIR  = OUTPUTS_DIR / "models"
PLOTS_DIR   = OUTPUTS_DIR / "plots"
RESULTS_DIR = OUTPUTS_DIR / "results"
for _d in (MODELS_DIR, PLOTS_DIR, RESULTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ── Epochs (reduce for smoke-test) ────────────────────────────────────────────
PRETRAIN_EPOCHS = 30
JOINT_EPOCHS    = 30
FINETUNE_EPOCHS = 15   # Phase 3: encoder unfreeze fine-tuning

# ── Classes ───────────────────────────────────────────────────────────────────
CLASS_MAP = {
    "NormalVideos": "Normal",
    "RoadAccidents": "RoadAccidents",
    "Shoplifting":   "Shoplifting",
    "Arson":        "Arson",
    "Burglary":     "Burglary",
}
CLASSES         = list(CLASS_MAP.values())
NUM_CLASSES     = len(CLASSES)
CLASS_TO_IDX    = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS    = {i: c for c, i in CLASS_TO_IDX.items()}
ANOMALY_CLASSES = [c for c in CLASSES if c != "Normal"]

# ── Data constants ────────────────────────────────────────────────────────────
FRAME_H      = 64
FRAME_W      = 64
CHANNELS     = 3        # decoder output channels (RGB)
IN_CHANNELS  = 6        # encoder input: RGB + frame-diff
CLIP_LEN     = 16
TRAIN_STRIDE = 16
TEST_STRIDE  = 8
MAX_FRAMES   = None

# ── Augmentation ──────────────────────────────────────────────────────────────
AUG_BRIGHTNESS = 0.2
AUG_CONTRAST   = 0.2
AUG_NOISE_STD  = 0.02

# ── Model ─────────────────────────────────────────────────────────────────────
LATENT_DIM  = 256
LSTM_UNITS  = 128
FC_UNITS    = 64
DROPOUT     = 0.65     # 0.6 → 0.65 for stronger regularisation
_SPATIAL    = FRAME_H // 8        # 8
_BOTTLENECK = _SPATIAL ** 2 * 128  # 8192

# ── Training — sized for 5-7 GB free VRAM on 2080 Ti ─────────────────────────
BATCH_SIZE    = 32    # raise to 64 if ≥8 GB VRAM is free
EVAL_BS       = 64
NUM_WORKERS   = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 5e-4
LAMBDA        = 0.5
FINETUNE_LAMBDA = 0.0   # Phase 3: pure CE — encoder adapts freely without reconstruction conflict
# Phase 3 fine-tune learning rates (encoder unlocked, differential LR)
FINETUNE_LR_ENC  = 1e-5   # slow — avoid destroying pretrained representations
FINETUNE_LR_HEAD = 1e-4   # faster — head adapts to unlocked encoder
SEED          = 42
VAL_SPLIT     = 0.25

print("Configuration loaded.")
print(f"  Dataset root : {DATASET_ROOT.resolve()}")
print(f"  Outputs      : {OUTPUTS_DIR.resolve()}")
print(f"  Batch size   : {BATCH_SIZE}  (eval {EVAL_BS})")
print(f"  Dropout      : {DROPOUT}")

## 1  Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run(
    f"{sys.executable} -m pip install -q "
    "torch torchvision pillow scikit-learn matplotlib seaborn tqdm psutil kaggle",
    shell=True, check=True
)
print("Packages ready.")

## 2  Download Dataset

**Option A — Kaggle API (recommended)**  
1. Go to https://www.kaggle.com/settings → Account → **Create New API Token**  
2. Place `kaggle.json` in `~/.kaggle/` (Linux) or `%USERPROFILE%\.kaggle\` (Windows)  
3. Uncomment and run the download block below.

**Option B — Manual**  
Download `ucf-crime-dataset.zip` from Kaggle and unzip so the structure is:  
```
data/dataset/Train/NormalVideos/*.png
data/dataset/Train/RoadAccidents/*.png  ...
data/dataset/Test/NormalVideos/*.png  ...
```
Then set `DATASET_ROOT` in Cell 0 to point to the `dataset/` folder.

In [ ]:
import os

# ── Uncomment to download via Kaggle API ──────────────────────────────────────
# os.makedirs(str(DATASET_ROOT.parent), exist_ok=True)
# os.system("mkdir -p ~/.kaggle && chmod 600 ~/.kaggle/kaggle.json")
# os.system(
#     f"kaggle datasets download -d odins0n/ucf-crime-dataset "
#     f"-p {DATASET_ROOT.parent} --unzip"
# )
# # The unzipped folder is usually named 'dataset'; rename if needed:
# # os.rename(str(DATASET_ROOT.parent / 'ucf-crime-dataset'), str(DATASET_ROOT))

# ── Verify ────────────────────────────────────────────────────────────────────
assert DATASET_ROOT.exists(), (
    f"Dataset not found: {DATASET_ROOT}\n"
    "Set DATASET_ROOT in Cell 0 or run the download block above."
)
train_dir = DATASET_ROOT / "Train"
test_dir  = DATASET_ROOT / "Test"
assert train_dir.exists(), f"Missing Train dir: {train_dir}"
assert test_dir.exists(),  f"Missing Test  dir: {test_dir}"

print(f"Dataset found at {DATASET_ROOT.resolve()}")
print(f"Train classes: {[d.name for d in sorted(train_dir.iterdir()) if d.is_dir()]}")
print(f"Test  classes: {[d.name for d in sorted(test_dir.iterdir())  if d.is_dir()]}")

## 3  Imports & Device

In [ ]:
import random, json
from collections import defaultdict, Counter

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_recall_fscore_support, f1_score,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.utils.class_weight import compute_class_weight
import psutil

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    total = p.total_memory / 1e9
    print(f"GPU     : {p.name}")
    print(f"VRAM    : {total:.1f} GB")
vm = psutil.virtual_memory()
print(f"RAM     : {vm.total/1e9:.1f} GB total  |  {vm.available/1e9:.1f} GB free")

## 4  Model Architecture

In [ ]:
class Encoder(nn.Module):
    """TimeDistributed CNN: (B, T, 6, H, W) -> (B, T, LATENT_DIM)"""
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(IN_CHANNELS, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),           nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),          nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Linear(_BOTTLENECK, LATENT_DIM), nn.ReLU()
        )
    def forward(self, x):
        B, T, C, H, W = x.shape
        return self.fc(self.cnn(x.view(B*T, C, H, W))).view(B, T, LATENT_DIM)


class Decoder(nn.Module):
    """TimeDistributed CNN decoder: (B, T, LATENT_DIM) -> (B, T, 3, H, W)"""
    def __init__(self):
        super().__init__()
        self.fc     = nn.Sequential(nn.Linear(LATENT_DIM, _BOTTLENECK), nn.ReLU())
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  nn.ReLU(),
            nn.ConvTranspose2d(32, CHANNELS, 4, stride=2, padding=1), nn.Sigmoid(),
        )
    def forward(self, x):
        B, T, _ = x.shape
        x = self.fc(x.view(B*T, LATENT_DIM)).view(B*T, 128, _SPATIAL, _SPATIAL)
        return self.deconv(x).view(B, T, CHANNELS, FRAME_H, FRAME_W)


class LSTMHead(nn.Module):
    """LSTM temporal classifier: (B, T, LATENT_DIM) -> (B, NUM_CLASSES)"""
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(LATENT_DIM, LSTM_UNITS, batch_first=True)
        self.fc   = nn.Sequential(
            nn.Linear(LSTM_UNITS, FC_UNITS), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(FC_UNITS, NUM_CLASSES),
        )
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h.squeeze(0))


class TransformerHead(nn.Module):
    """Self-attention temporal classifier: (B, T, LATENT_DIM) -> (B, NUM_CLASSES).
    Two-layer transformer encoder with learnable positional embeddings.
    norm_first=True (Pre-LN) gives more stable gradients on short sequences."""
    def __init__(self):
        super().__init__()
        self.pos_embed = nn.Embedding(CLIP_LEN, LATENT_DIM)
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=LATENT_DIM, nhead=4, dim_feedforward=512,
            dropout=DROPOUT, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Sequential(
            nn.Linear(LATENT_DIM, FC_UNITS), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(FC_UNITS, NUM_CLASSES),
        )
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x   = x + self.pos_embed(pos)
        x   = self.transformer(x).mean(dim=1)   # temporal mean-pool
        return self.fc(x)


class TemporalPoolHead(nn.Module):
    """Mean-pool ablation baseline."""
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(LATENT_DIM, FC_UNITS), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(FC_UNITS, NUM_CLASSES),
        )
    def forward(self, x): return self.fc(x.mean(dim=1))


class Autoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder; self.decoder = decoder
    def forward(self, x): return self.decoder(self.encoder(x))


class JointModel(nn.Module):
    def __init__(self, encoder, decoder, temporal_head=None):
        super().__init__()
        self.encoder       = encoder
        self.decoder       = decoder
        self.temporal_head = temporal_head if temporal_head is not None else LSTMHead()
    def forward(self, x):
        emb = self.encoder(x)
        return self.decoder(emb), self.temporal_head(emb)


class SingleFrameCNN(nn.Module):
    """Ablation variant A: classify the middle RGB frame only."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(CHANNELS, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),        nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),       nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(_BOTTLENECK, 256), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(256, NUM_CLASSES),
        )
    def forward(self, x): return self.net(x)


class SingleFrameWrapper(nn.Module):
    """Makes SingleFrameCNN accept (B,T,C,H,W) clips for eval compatibility."""
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, x): return None, self.model(x[:, x.shape[1]//2, :3])


def freeze_decoder(decoder):
    for p in decoder.parameters(): p.requires_grad = False
    print("[INFO] Decoder frozen.")


enc_t = Encoder()
print(f"Encoder params        : {sum(p.numel() for p in enc_t.parameters()):,}")
dec_t = Decoder()
print(f"Decoder params        : {sum(p.numel() for p in dec_t.parameters()):,}")
lst_t = LSTMHead()
print(f"LSTMHead params       : {sum(p.numel() for p in lst_t.parameters()):,}")
atn_t = TransformerHead()
print(f"TransformerHead params: {sum(p.numel() for p in atn_t.parameters()):,}")
del enc_t, dec_t, lst_t, atn_t

## 5  Data Pipeline (Disk-Based)

In [ ]:
def _get_video_id(filename):
    return Path(filename).stem.rsplit("_", 1)[0]

def _get_frame_number(filename):
    return int(Path(filename).stem.rsplit("_", 1)[-1])

def discover_videos(data_root, split):
    split_dir = Path(data_root) / split
    result = {}
    for folder_name, label in CLASS_MAP.items():
        folder = split_dir / folder_name
        if not folder.exists():
            print(f"[WARN] Not found: {folder}"); continue
        groups = defaultdict(list)
        for f in folder.glob("*.png"):
            groups[_get_video_id(f.name)].append(f)
        for vid in groups:
            groups[vid].sort(key=lambda p: _get_frame_number(p.name))
        result[label] = dict(groups)
    return result

def build_clips(video_groups, stride, max_frames=None):
    clips = []; frame_count = 0
    vids = list(video_groups.keys()); random.shuffle(vids)
    for vid in vids:
        frames = video_groups[vid]
        if max_frames and frame_count >= max_frames: break
        for start in range(0, len(frames) - CLIP_LEN + 1, stride):
            clips.append(frames[start: start + CLIP_LEN])
            frame_count += CLIP_LEN
            if max_frames and frame_count >= max_frames: break
    return clips

def make_video_splits(video_dict_train, video_dict_test, val_ratio=VAL_SPLIT):
    train_paths, y_train = [], []
    val_paths,   y_val   = [], []
    normal_train_vids    = {}
    print("Train / Val clips:")
    for label, video_groups in video_dict_train.items():
        cls_idx = CLASS_TO_IDX[label]
        vid_ids = sorted(video_groups.keys()); random.shuffle(vid_ids)
        n_val   = max(1, round(len(vid_ids) * val_ratio))
        val_v   = {v: video_groups[v] for v in vid_ids[:n_val]}
        tr_v    = {v: video_groups[v] for v in vid_ids[n_val:]}
        if label == "Normal": normal_train_vids = tr_v
        # Use TEST_STRIDE for anomaly classes to double minority-class clips
        stride = TRAIN_STRIDE if label == "Normal" else TEST_STRIDE
        tr = build_clips(tr_v, stride, MAX_FRAMES)
        va = build_clips(val_v, stride, MAX_FRAMES)
        train_paths.extend(tr); y_train.extend([cls_idx]*len(tr))
        val_paths.extend(va);   y_val.extend([cls_idx]*len(va))
        print(f"  {label:<12} train={len(tr):>5}  val={len(va):>4}  ({len(vid_ids)} vids)")
    test_paths, y_test = [], []
    print("Test clips:")
    for label, video_groups in video_dict_test.items():
        cls_idx = CLASS_TO_IDX[label]
        clips   = build_clips(video_groups, TEST_STRIDE, MAX_FRAMES)
        test_paths.extend(clips); y_test.extend([cls_idx]*len(clips))
        print(f"  {label:<12} test ={len(clips):>5}")
    return (train_paths, np.array(y_train, np.int32),
            val_paths,   np.array(y_val,   np.int32),
            test_paths,  np.array(y_test,  np.int32),
            normal_train_vids)

def _augment_rgb(rgb, normalize_brightness=False):
    """Spatial + colour augmentation applied consistently across all T frames."""
    rgb = rgb.copy()
    # Horizontal flip
    if random.random() < 0.5:
        rgb = rgb[:, :, ::-1, :]
    # Brightness shift
    if random.random() < 0.5:
        rgb = np.clip(rgb + random.uniform(-AUG_BRIGHTNESS, AUG_BRIGHTNESS), 0.0, 1.0)
    # Contrast stretch
    if random.random() < 0.5:
        f = random.uniform(1-AUG_CONTRAST, 1+AUG_CONTRAST)
        rgb = np.clip(rgb.mean() + f*(rgb - rgb.mean()), 0.0, 1.0)
    # Gaussian noise
    if random.random() < 0.3:
        rgb = np.clip(rgb + np.random.normal(0, AUG_NOISE_STD, rgb.shape).astype(np.float32), 0.0, 1.0)
    # Saturation jitter — scales chroma while preserving luminance (new)
    if random.random() < 0.4:
        f    = random.uniform(0.7, 1.3)
        gray = (0.299*rgb[...,0] + 0.587*rgb[...,1] + 0.114*rgb[...,2])[..., np.newaxis]
        rgb  = np.clip(gray + f * (rgb - gray), 0.0, 1.0).astype(np.float32)
    # Cutout — erase a random patch; forces model to use spatial context (new)
    if random.random() < 0.3:
        cy, cx = random.randint(8, FRAME_H-8), random.randint(8, FRAME_W-8)
        bh, bw = random.randint(8, 18), random.randint(8, 18)
        y1 = max(0, cy-bh//2); y2 = min(FRAME_H, cy+bh//2)
        x1 = max(0, cx-bw//2); x2 = min(FRAME_W, cx+bw//2)
        rgb[:, y1:y2, x1:x2, :] = rgb.mean()   # fill with clip mean, not black
    # Brightness normalisation (only when explicitly requested)
    if normalize_brightness and random.random() < 0.4:
        clip_mean = rgb.mean()
        if clip_mean > 0.05:
            target = random.uniform(0.2, 0.5)
            rgb = np.clip(rgb * (target / clip_mean), 0.0, 1.0)
    return rgb

def _rgb_to_6ch(rgb):
    diffs = np.diff(rgb, axis=0)
    diffs = np.concatenate([np.zeros_like(rgb[:1]), diffs], axis=0)
    return np.concatenate([rgb, diffs], axis=-1)  # (T, H, W, 6)

def _load_frames(clip_paths):
    """Load one clip from disk -> (T, H, W, 3) float32 in [0,1]."""
    frames = []
    for p in clip_paths:
        img = Image.open(p).convert("RGB")
        if img.size != (FRAME_W, FRAME_H):
            img = img.resize((FRAME_W, FRAME_H), Image.BILINEAR)
        frames.append(np.array(img, dtype=np.float32) / 255.0)
    return np.stack(frames)

def _load_sample_clips_float(clip_paths_list, n=4):
    """Load n random clips as (n, T, H, W, 3) float32 for visualisation."""
    idx = np.random.choice(len(clip_paths_list), min(n, len(clip_paths_list)), replace=False)
    return np.stack([_load_frames(clip_paths_list[i]) for i in idx])


class ClipDataset(Dataset):
    """
    Disk-based dataset.  Each __getitem__ reads CLIP_LEN PNGs.
    Returns (x, label) when y is provided, just x otherwise.
    x shape: (T, 6, H, W) float32
    """
    def __init__(self, clips, y=None, training=False, normalize_brightness=False):
        self.clips                = clips
        self.y                    = y
        self.training             = training
        self.normalize_brightness = normalize_brightness

    def __len__(self): return len(self.clips)

    def __getitem__(self, idx):
        rgb = _load_frames(self.clips[idx])            # (T, H, W, 3)
        if self.training:
            # Temporal reverse — free motion-direction invariance, no extra data needed
            if random.random() < 0.3:
                rgb = rgb[::-1]
            rgb = _augment_rgb(rgb, normalize_brightness=self.normalize_brightness)
        x6  = _rgb_to_6ch(rgb)                        # (T, H, W, 6)
        x   = torch.from_numpy(
            np.ascontiguousarray(x6.transpose(0, 3, 1, 2))
        ).float()                                      # (T, 6, H, W)
        if self.y is not None:
            return x, torch.tensor(int(self.y[idx]), dtype=torch.long)
        return x


class MidFrameDataset(ClipDataset):
    """Returns middle RGB frame only — for ablation variant A."""
    def __getitem__(self, idx):
        x, y = super().__getitem__(idx)
        return x[CLIP_LEN // 2, :3], y  # (3, H, W)


def make_loader(clips, y=None, shuffle=True, training=False,
                bs=BATCH_SIZE, workers=NUM_WORKERS,
                normalize_brightness=False):
    ds = ClipDataset(clips, y, training=training,
                     normalize_brightness=normalize_brightness)
    sampler = None
    if training and y is not None:
        # Balance classes at batch level — minority anomaly classes get equal
        # representation as Normal instead of being swamped 82:18 per batch.
        counts = np.bincount(y, minlength=NUM_CLASSES).astype(float)
        counts = np.maximum(counts, 1)
        weights = torch.from_numpy(1.0 / counts[y]).float()
        # 5000 per class: balanced sampling across all 5 classes
        sampler = WeightedRandomSampler(weights, num_samples=5000 * NUM_CLASSES, replacement=True)
        shuffle = False   # mutually exclusive with sampler
    return DataLoader(
        ds, batch_size=bs, shuffle=shuffle, sampler=sampler,
        num_workers=workers, pin_memory=True,
        persistent_workers=(workers > 0),
    )

print("Data pipeline helpers ready.")

## 6  Training Functions

In [ ]:
def pretrain_autoencoder(autoencoder, normal_paths, val_split=0.15, device=DEVICE):
    n_val    = int(len(normal_paths) * val_split)
    tr_paths = normal_paths[n_val:]
    va_paths = normal_paths[:n_val]
    tr_loader = make_loader(tr_paths, y=None, shuffle=True, training=True)
    va_loader = make_loader(va_paths, y=None, shuffle=False)

    autoencoder = autoencoder.to(device)
    opt   = torch.optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    crit  = nn.MSELoss()
    best  = float("inf"); pat = 0
    hist  = {"train_loss": [], "val_loss": []}

    for epoch in range(PRETRAIN_EPOCHS):
        autoencoder.train(); tr_l = []
        for x in tr_loader:
            x = x.to(device); opt.zero_grad()
            loss = crit(autoencoder(x), x[:, :, :3, :, :])
            loss.backward(); opt.step(); tr_l.append(loss.item())
        autoencoder.eval(); va_l = []
        with torch.no_grad():
            for x in va_loader:
                x = x.to(device)
                va_l.append(crit(autoencoder(x), x[:, :, :3, :, :]).item())
        tl = float(np.mean(tr_l)); vl = float(np.mean(va_l))
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        sched.step(vl)
        if vl < best:
            best = vl; pat = 0
            torch.save(autoencoder.state_dict(), MODELS_DIR / "autoencoder_best.pth")
        else:
            pat += 1
            if pat >= 7: print(f"Early stop @ {epoch+1}"); break
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{PRETRAIN_EPOCHS}  loss={tl:.5f}  val={vl:.5f}")

    autoencoder.load_state_dict(
        torch.load(MODELS_DIR / "autoencoder_best.pth", map_location=device)
    )
    torch.save(autoencoder.encoder.state_dict(), MODELS_DIR / "encoder_pretrained.pth")
    torch.save(autoencoder.decoder.state_dict(), MODELS_DIR / "decoder_pretrained.pth")
    print(f"Pre-training done. Best val loss: {best:.5f}")
    return hist


def joint_train(joint_model, train_paths, y_train, val_paths, y_val,
                lam=LAMBDA, device=DEVICE, class_weights=None,
                save_path=None):
    # Same augmentation as pretraining so frozen encoder sees in-distribution inputs.
    # Brightness normalisation removed: frozen encoder was pretrained without it,
    # so enabling it here creates a train/val feature mismatch (val_acc starts ~0.10).
    ckpt = save_path if save_path is not None else MODELS_DIR / "joint_model_best.pth"
    tr_loader = make_loader(train_paths, y_train, shuffle=True, training=True)
    va_loader = make_loader(val_paths,   y_val,   shuffle=False)

    joint_model = joint_model.to(device)
    for p in joint_model.encoder.parameters(): p.requires_grad = False
    trainable = list(joint_model.temporal_head.parameters())
    opt   = torch.optim.Adam(trainable, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=JOINT_EPOCHS)
    mse   = nn.MSELoss()
    wt    = (torch.tensor(class_weights, dtype=torch.float32).to(device)
             if class_weights is not None else None)
    ce    = nn.CrossEntropyLoss(weight=wt, label_smoothing=0.05)
    best_f1 = 0.0; no_imp = 0
    hist  = {k: [] for k in ["train_loss","train_recon","train_cls","train_acc",
                               "val_loss","val_recon","val_cls","val_acc","val_macro_f1"]}

    for epoch in range(JOINT_EPOCHS):
        joint_model.train(); joint_model.decoder.eval()
        tr = {"l":[],"r":[],"c":[]}; tc = tt = 0
        for x, labels in tr_loader:
            x, labels = x.to(device), labels.to(device); opt.zero_grad()
            recon, logits = joint_model(x)
            r = mse(recon, x[:,:,:3,:,:]); c = ce(logits, labels)
            loss = r + lam*c; loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); opt.step()
            tr["l"].append(loss.item()); tr["r"].append(r.item()); tr["c"].append(c.item())
            tc += (logits.argmax(1)==labels).sum().item(); tt += len(labels)

        joint_model.eval(); va = {"l":[],"r":[],"c":[]}; vc = vt = 0
        per_cls_correct = [0]*NUM_CLASSES; per_cls_total = [0]*NUM_CLASSES
        all_val_preds = []; all_val_labels = []
        with torch.no_grad():
            for x, labels in va_loader:
                x, labels = x.to(device), labels.to(device)
                recon, logits = joint_model(x)
                r = mse(recon, x[:,:,:3,:,:]); c = ce(logits, labels)
                va["l"].append((r+lam*c).item()); va["r"].append(r.item()); va["c"].append(c.item())
                preds = logits.argmax(1)
                vc += (preds==labels).sum().item(); vt += len(labels)
                all_val_preds.extend(preds.cpu().tolist())
                all_val_labels.extend(labels.cpu().tolist())
                for cls_i in range(NUM_CLASSES):
                    mask = labels == cls_i
                    per_cls_correct[cls_i] += (preds[mask]==labels[mask]).sum().item()
                    per_cls_total[cls_i]   += mask.sum().item()

        tl=float(np.mean(tr["l"])); vl=float(np.mean(va["l"]))
        ta=tc/tt; va_acc=vc/vt
        val_macro_f1 = f1_score(all_val_labels, all_val_preds,
                                average="macro", zero_division=0)
        for k,v in zip(["train_loss","train_recon","train_cls","train_acc",
                         "val_loss","val_recon","val_cls","val_acc","val_macro_f1"],
                        [tl,np.mean(tr["r"]),np.mean(tr["c"]),ta,
                         vl,np.mean(va["r"]),np.mean(va["c"]),va_acc,val_macro_f1]):
            hist[k].append(float(v))
        sched.step()
        # Checkpoint on macro F1 — val_acc is Normal-biased (82% of val set is Normal)
        if val_macro_f1 > best_f1:
            best_f1 = val_macro_f1; no_imp = 0
            torch.save(joint_model.state_dict(), ckpt)
        else:
            no_imp += 1
            if no_imp >= 10: print(f"Early stop @ {epoch+1}"); break
        per_cls_str = "  ".join(
            f"{IDX_TO_CLASS[i][0]}:{per_cls_correct[i]/max(per_cls_total[i],1):.2f}"
            for i in range(NUM_CLASSES)
        )
        print(f"Epoch {epoch+1:02d}/{JOINT_EPOCHS}  "
              f"loss={tl:.4f} cls={np.mean(tr['c']):.4f} acc={ta:.3f} | "
              f"val_acc={va_acc:.3f}  f1={val_macro_f1:.3f}  [{per_cls_str}]")

    joint_model.load_state_dict(torch.load(ckpt, map_location=device))
    print(f"Joint training done. Best val macro F1: {best_f1:.4f}")
    return hist


def finetune_encoder(joint_model, train_paths, y_train, val_paths, y_val,
                     lam=LAMBDA, device=DEVICE, class_weights=None,
                     save_path=None):
    """Phase 3: unlock encoder and fine-tune end-to-end with differential LRs."""
    ckpt = save_path if save_path is not None else MODELS_DIR / "joint_model_finetuned.pth"
    tr_loader = make_loader(train_paths, y_train, shuffle=True, training=True)
    va_loader = make_loader(val_paths,   y_val,   shuffle=False)

    joint_model = joint_model.to(device)
    for p in joint_model.encoder.parameters(): p.requires_grad = True

    opt = torch.optim.Adam([
        {"params": joint_model.encoder.parameters(),       "lr": FINETUNE_LR_ENC},
        {"params": joint_model.temporal_head.parameters(), "lr": FINETUNE_LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FINETUNE_EPOCHS)
    mse   = nn.MSELoss()
    wt    = (torch.tensor(class_weights, dtype=torch.float32).to(device)
             if class_weights is not None else None)
    ce    = nn.CrossEntropyLoss(weight=wt, label_smoothing=0.05)
    best_f1 = 0.0; no_imp = 0
    hist  = {k: [] for k in ["train_loss","train_acc","val_loss","val_acc","val_macro_f1"]}

    for epoch in range(FINETUNE_EPOCHS):
        joint_model.train()
        tr_l = []; tc = tt = 0
        for x, labels in tr_loader:
            x, labels = x.to(device), labels.to(device); opt.zero_grad()
            recon, logits = joint_model(x)
            r = mse(recon, x[:,:,:3,:,:]); c = ce(logits, labels)
            loss = r + lam*c; loss.backward()
            torch.nn.utils.clip_grad_norm_(joint_model.parameters(), 1.0); opt.step()
            tr_l.append(loss.item())
            tc += (logits.argmax(1)==labels).sum().item(); tt += len(labels)
        sched.step()

        joint_model.eval(); va_l = []; vc = vt = 0
        per_cls_correct = [0]*NUM_CLASSES; per_cls_total = [0]*NUM_CLASSES
        all_val_preds = []; all_val_labels = []
        with torch.no_grad():
            for x, labels in va_loader:
                x, labels = x.to(device), labels.to(device)
                recon, logits = joint_model(x)
                r = mse(recon, x[:,:,:3,:,:]); c = ce(logits, labels)
                va_l.append((r+lam*c).item())
                preds = logits.argmax(1)
                vc += (preds==labels).sum().item(); vt += len(labels)
                all_val_preds.extend(preds.cpu().tolist())
                all_val_labels.extend(labels.cpu().tolist())
                for cls_i in range(NUM_CLASSES):
                    mask = labels == cls_i
                    per_cls_correct[cls_i] += (preds[mask]==labels[mask]).sum().item()
                    per_cls_total[cls_i]   += mask.sum().item()

        tl=float(np.mean(tr_l)); vl=float(np.mean(va_l))
        ta=tc/tt; va_acc=vc/vt
        val_macro_f1 = f1_score(all_val_labels, all_val_preds,
                                average="macro", zero_division=0)
        hist["train_loss"].append(tl); hist["train_acc"].append(ta)
        hist["val_loss"].append(vl);   hist["val_acc"].append(va_acc)
        hist["val_macro_f1"].append(val_macro_f1)
        if val_macro_f1 > best_f1:
            best_f1 = val_macro_f1; no_imp = 0
            torch.save(joint_model.state_dict(), ckpt)
        else:
            no_imp += 1
            if no_imp >= 5: print(f"Early stop @ {epoch+1}"); break
        per_cls_str = "  ".join(
            f"{IDX_TO_CLASS[i][0]}:{per_cls_correct[i]/max(per_cls_total[i],1):.2f}"
            for i in range(NUM_CLASSES)
        )
        print(f"FT {epoch+1:02d}/{FINETUNE_EPOCHS}  "
              f"loss={tl:.4f} acc={ta:.3f} | val={vl:.4f} val_acc={va_acc:.3f} "
              f"f1={val_macro_f1:.3f}  [{per_cls_str}]")

    joint_model.load_state_dict(torch.load(ckpt, map_location=device))
    print(f"Fine-tuning done. Best val macro F1: {best_f1:.4f}")
    return hist


def plot_loss(history, title, save_path):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(history["train_loss"], label="Train", lw=1.5)
    ax.plot(history["val_loss"],   label="Val",   lw=1.5, ls="--")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title(title, fontweight="bold"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"[OK] {save_path}")

def plot_joint_loss(history, save_path):
    """Three-panel: loss / accuracy / macro-F1 (checkpoint metric)."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 3.5))
    ax1.plot(history["train_loss"], label="Train", lw=1.5)
    ax1.plot(history["val_loss"],   label="Val",   lw=1.5, ls="--")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("Joint Training Loss", fontweight="bold")
    ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history["train_acc"], label="Train acc", lw=1.5)
    ax2.plot(history["val_acc"],   label="Val acc",   lw=1.5, ls="--")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_title("Classification Accuracy", fontweight="bold")
    ax2.legend(); ax2.grid(alpha=0.3)
    if "val_macro_f1" in history:
        ax3.plot(history["val_macro_f1"], label="Val macro F1", lw=1.5, color="darkorange")
        best_ep = int(np.argmax(history["val_macro_f1"]))
        ax3.axvline(best_ep, color="gray", ls=":", lw=1, label=f"Best ep={best_ep+1}")
        ax3.set_xlabel("Epoch"); ax3.set_ylabel("Macro F1")
        ax3.set_title("Val Macro F1 (checkpoint metric)", fontweight="bold")
        ax3.legend(); ax3.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)
    print(f"[OK] {save_path}")

## 7  Evaluation Functions

In [ ]:
_CLASS_COLORS = {
    "Normal":"#4FC3F7","RoadAccidents":"#29B6F6",
    "Shoplifting":"#66BB6A","Arson":"#FF7043","Burglary":"#AB47BC",
}

def _save_json(data, filename):
    out = RESULTS_DIR / filename
    with open(out, "w") as f: json.dump(data, f, indent=2, default=float)
    print(f"[OK] {out}")

def _make_loader_unlabeled(paths, bs=EVAL_BS):
    return DataLoader(ClipDataset(paths, None, training=False),
                      batch_size=bs, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

def compute_reconstruction_errors(model, paths, bs=EVAL_BS, device=DEVICE):
    model.eval(); errors = []
    with torch.no_grad():
        for x in _make_loader_unlabeled(paths, bs):
            x = x.to(device)
            out = model(x)
            recon = out[0] if isinstance(out, tuple) else out
            mse = ((recon - x[:,:,:3,:,:]) ** 2).mean(dim=(1,2,3,4))
            errors.extend(mse.cpu().tolist())
    return np.array(errors)

def compute_threshold(errors, n_std=2.0):
    return float(errors.mean() + n_std * errors.std())

def find_optimal_threshold(errors, binary_labels):
    fpr, tpr, thresholds = roc_curve(binary_labels, errors)
    return float(thresholds[np.argmax(tpr - fpr)])

def threshold_comparison(model, val_normal_paths, test_paths, y_test, device=DEVICE):
    normal_err = compute_reconstruction_errors(model, val_normal_paths, device=device)
    test_err   = compute_reconstruction_errors(model, test_paths,       device=device)
    binary_gt  = (y_test != CLASS_TO_IDX["Normal"]).astype(int)
    auc        = roc_auc_score(binary_gt, test_err)
    results = {}
    for name, thr in [("Mean+2std", compute_threshold(normal_err, 2.0)),
                      ("Mean+3std", compute_threshold(normal_err, 3.0)),
                      ("ROC-Opt",   find_optimal_threshold(test_err, binary_gt))]:
        preds = (test_err > thr).astype(int)
        p,r,f,_ = precision_recall_fscore_support(binary_gt, preds, average="binary", zero_division=0)
        results[name] = {"threshold":thr,"precision":p,"recall":r,"f1":f,"auc":auc}
    print(f"\n{'Method':<12} {'Threshold':>10} {'Precision':>10} {'Recall':>8} {'F1':>8} {'AUC':>8}")
    print("-"*58)
    for n,r in results.items():
        print(f"{n:<12} {r['threshold']:>10.5f} {r['precision']:>10.3f} {r['recall']:>8.3f} {r['f1']:>8.3f} {r['auc']:>8.3f}")
    _save_json(results, "threshold_comparison.json")
    return results, normal_err, test_err, binary_gt

def evaluate_classifier(model, paths, y, bs=EVAL_BS, device=DEVICE):
    model.eval(); preds = []
    loader = DataLoader(ClipDataset(paths, y, training=False),
                        batch_size=bs, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=(NUM_WORKERS > 0))
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            out = model(x)
            logits = out[1] if isinstance(out, tuple) else out
            preds.extend(logits.argmax(1).cpu().tolist())
    y_pred  = np.array(preds)
    report  = classification_report(y, y_pred, target_names=CLASSES,
                                    output_dict=True, zero_division=0)
    print(classification_report(y, y_pred, target_names=CLASSES, zero_division=0))
    _save_json(report, "classification_report.json")
    return y_pred, report

def evaluate_binary(model, paths, y, bs=EVAL_BS, device=DEVICE):
    model.eval(); all_probs = []
    loader = DataLoader(ClipDataset(paths, y, training=False),
                        batch_size=bs, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=(NUM_WORKERS > 0))
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            out = model(x)
            logits = out[1] if isinstance(out, tuple) else out
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    probs    = np.concatenate(all_probs, axis=0)
    p_anom   = 1.0 - probs[:, CLASS_TO_IDX["Normal"]]
    binary_gt = (y != CLASS_TO_IDX["Normal"]).astype(int)
    pred_bin  = (p_anom > 0.5).astype(int)
    auc = roc_auc_score(binary_gt, p_anom)
    p,r,f,_ = precision_recall_fscore_support(binary_gt, pred_bin, average="binary", zero_division=0)
    acc = float((pred_bin == binary_gt).mean())
    result = {"accuracy":acc,"precision":p,"recall":r,"f1":f,"auc":auc}
    print(f"\nBinary (Normal vs Anomaly): Acc={acc:.3f} P={p:.3f} R={r:.3f} F1={f:.3f} AUC={auc:.3f}")
    _save_json(result, "binary_anomaly_report.json")
    return result, p_anom, binary_gt

def compute_fusion_score(recon_errors, class_probs, alpha=0.5):
    norm_r = (recon_errors - recon_errors.min()) / (recon_errors.ptp() + 1e-8)
    p_anom = 1.0 - class_probs[:, CLASS_TO_IDX["Normal"]]
    return alpha * norm_r + (1.0 - alpha) * p_anom

def get_embeddings(encoder, paths, bs=EVAL_BS, device=DEVICE):
    encoder.eval(); embs = []
    with torch.no_grad():
        for x in _make_loader_unlabeled(paths, bs):
            embs.append(encoder(x.to(device)).mean(dim=1).cpu().numpy())
    return np.concatenate(embs, axis=0)

# ── Plot helpers ──────────────────────────────────────────────────────────────
def plot_roc_curve(errors, binary_gt, save_path):
    fpr,tpr,_ = roc_curve(binary_gt, errors); auc = roc_auc_score(binary_gt, errors)
    fig,ax = plt.subplots(figsize=(5,4))
    ax.plot(fpr,tpr,lw=2,label=f"AUC={auc:.3f}"); ax.plot([0,1],[0,1],"k--",lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("ROC — Anomaly Detection",fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"[OK] {save_path}")

def plot_anomaly_scores(errors, y, threshold, save_path):
    fig,ax = plt.subplots(figsize=(12,3.5)); x=np.arange(len(errors))
    nm=y==CLASS_TO_IDX["Normal"]; an=~nm
    ax.scatter(x[nm],errors[nm],s=6,alpha=0.5,color="steelblue",label="Normal")
    ax.scatter(x[an],errors[an],s=6,alpha=0.7,color="crimson",  label="Anomaly")
    ax.axhline(threshold,color="darkorange",ls="--",lw=1.5,label=f"Thr={threshold:.4f}")
    ax.set_xlabel("Clip"); ax.set_ylabel("MSE")
    ax.set_title("Per-Clip Anomaly Scores",fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout()
    fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"[OK] {save_path}")

def plot_confusion_matrix(y_true, y_pred, save_path):
    cm_   = confusion_matrix(y_true, y_pred)
    cm_n  = cm_.astype(float) / cm_.sum(axis=1,keepdims=True)
    fig,axes = plt.subplots(1,2,figsize=(12,4.5))
    for ax,data,fmt,title in [(axes[0],cm_,"d","Counts"),(axes[1],cm_n,".2f","Normalised")]:
        sns.heatmap(data,annot=True,fmt=fmt,cmap="Blues",
                    xticklabels=CLASSES,yticklabels=CLASSES,ax=ax)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(title,fontweight="bold")
        ax.tick_params(axis="x",rotation=30,labelsize=8); ax.tick_params(axis="y",rotation=0,labelsize=8)
    plt.tight_layout(); fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig)
    print(f"[OK] {save_path}")

def plot_reconstruction_samples(model, sample_np, save_path=None, device=DEVICE):
    """sample_np: (n, T, H, W, 3) float32 loaded via _load_sample_clips_float"""
    n  = len(sample_np)
    x6 = np.stack([_rgb_to_6ch(sample_np[i]) for i in range(n)])  # (n,T,H,W,6)
    batch = torch.from_numpy(x6.transpose(0,1,4,2,3)).float().to(device)
    model.eval()
    with torch.no_grad():
        out = model(batch)
        recon = out[0] if isinstance(out, tuple) else out
    recon_np = recon.cpu().numpy().transpose(0,1,3,4,2)  # (n,T,H,W,3)
    fig,axes = plt.subplots(n, 2*CLIP_LEN, figsize=(CLIP_LEN*3, n*2))
    for row in range(n):
        for col in range(CLIP_LEN):
            axes[row][col].imshow(sample_np[row,col]); axes[row][col].axis("off")
            if row==0: axes[row][col].set_title(f"In {col+1}",fontsize=7)
            axes[row][CLIP_LEN+col].imshow(np.clip(recon_np[row,col],0,1))
            axes[row][CLIP_LEN+col].axis("off")
            if row==0: axes[row][CLIP_LEN+col].set_title(f"Rc {col+1}",fontsize=7)
    plt.suptitle("Input vs Reconstructed",fontweight="bold"); plt.tight_layout()
    if save_path: fig.savefig(save_path,dpi=150,bbox_inches="tight"); print(f"[OK] {save_path}")
    plt.show(); plt.close(fig)

def plot_mse_distribution(model, normal_paths, anomaly_paths, save_path, device=DEVICE):
    ne = compute_reconstruction_errors(model, normal_paths, device=device)
    ae = compute_reconstruction_errors(model, anomaly_paths, device=device)
    fig,ax = plt.subplots(figsize=(7,4))
    ax.hist(ne,bins=60,alpha=0.6,color="steelblue",label="Normal",density=True)
    ax.hist(ae,bins=60,alpha=0.6,color="crimson",  label="Anomaly",density=True)
    ax.set_xlabel("MSE"); ax.set_ylabel("Density")
    ax.set_title("MSE Distribution",fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"[OK] {save_path}")

def plot_latent_space(embeddings, labels, method="pca", save_path=None):
    reducer = (TSNE(n_components=2,random_state=42,perplexity=30)
               if method=="tsne" else PCA(n_components=2,random_state=42))
    coords  = reducer.fit_transform(embeddings)
    fig,ax  = plt.subplots(figsize=(7,5))
    for i,cls in enumerate(CLASSES):
        mask=labels==i
        ax.scatter(coords[mask,0],coords[mask,1],s=12,alpha=0.6,
                   color=_CLASS_COLORS.get(cls,"#888"),label=cls)
    ax.set_title(f"{'t-SNE' if method=='tsne' else 'PCA'} of Latent Embeddings",fontweight="bold")
    ax.legend(fontsize=9,markerscale=2); ax.grid(alpha=0.3); plt.tight_layout()
    if save_path: fig.savefig(save_path,dpi=150,bbox_inches="tight"); print(f"[OK] {save_path}")
    plt.show(); plt.close("all")

def plot_binary_roc(p_anom, binary_gt, save_path):
    fpr,tpr,_=roc_curve(binary_gt,p_anom); auc=roc_auc_score(binary_gt,p_anom)
    fig,ax=plt.subplots(figsize=(5,4))
    ax.plot(fpr,tpr,lw=2,label=f"AUC={auc:.3f}"); ax.plot([0,1],[0,1],"k--",lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("Binary Anomaly ROC",fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"[OK] {save_path}")

def plot_per_class_metrics(report, save_path):
    metrics={cls:report[cls] for cls in CLASSES if cls in report}
    x=np.arange(len(metrics)); w=0.25
    prec=[metrics[c]["precision"] for c in metrics]
    rec= [metrics[c]["recall"]    for c in metrics]
    f1=  [metrics[c]["f1-score"]  for c in metrics]
    fig,ax=plt.subplots(figsize=(9,4))
    ax.bar(x-w,prec,w,label="Precision",color="#1E88E5")
    ax.bar(x,  rec, w,label="Recall",   color="#43A047")
    ax.bar(x+w,f1,  w,label="F1",       color="#FB8C00")
    ax.set_xticks(x); ax.set_xticklabels(list(metrics.keys()),rotation=20,ha="right")
    ax.set_ylim(0,1.1); ax.set_ylabel("Score")
    ax.set_title("Per-Class Metrics",fontweight="bold")
    ax.legend(); ax.grid(axis="y",alpha=0.3); plt.tight_layout()
    fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"[OK] {save_path}")

print("Evaluation helpers ready.")

## 8  Timeline Functions

In [ ]:
TL_COLORS = {
    "Normal":"#4FC3F7","RoadAccidents":"#29B6F6",
    "Shoplifting":"#66BB6A","Arson":"#FF7043","Burglary":"#AB47BC",
}

def build_test_sequence(test_paths, y_test, clips_per_segment=20):
    """Interleave Normal and anomaly clips into a demo sequence (paths-based)."""
    seq_paths, seq_labels, segments = [], [], []
    normal_idx = np.where(y_test == CLASS_TO_IDX["Normal"])[0].copy()
    np.random.shuffle(normal_idx)

    def _add_normal(n):
        for i in normal_idx[:n]:
            seq_paths.append(test_paths[i]); seq_labels.append(CLASS_TO_IDX["Normal"])
        segments.append(("Normal", len(seq_paths)-n, len(seq_paths)-1))

    _add_normal(clips_per_segment)
    for cls in CLASSES:
        if cls == "Normal": continue
        idx  = CLASS_TO_IDX[cls]
        pool = np.where(y_test == idx)[0]
        if len(pool) == 0: continue
        start = len(seq_paths)
        for i in pool[:clips_per_segment]:
            seq_paths.append(test_paths[i]); seq_labels.append(idx)
        segments.append((cls, start, len(seq_paths)-1))
        _add_normal(clips_per_segment)

    return seq_paths, np.array(seq_labels, np.int32), segments

def sliding_window_predict(model, seq_paths, bs=64, device=DEVICE):
    model.eval(); pred_labels, pred_probs, recon_errors = [], [], []
    loader = DataLoader(
        ClipDataset(seq_paths, None, training=False),
        batch_size=bs, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )
    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            recon, logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            pred_labels.extend(probs.argmax(1).tolist())
            pred_probs.extend(probs.tolist())
            mse = ((recon - x[:,:,:3,:,:]) ** 2).mean(dim=(1,2,3,4)).cpu().numpy()
            recon_errors.extend(mse.tolist())
    return np.array(pred_labels), np.array(pred_probs), np.array(recon_errors)

def majority_vote_smooth(labels, window=3):
    smoothed = labels.copy(); half = window//2
    for i in range(len(labels)):
        lo=max(0,i-half); hi=min(len(labels),i+half+1)
        smoothed[i] = Counter(labels[lo:hi]).most_common(1)[0][0]
    return smoothed

def merge_segments(labels, recon_errors, threshold, fps=1.0):
    segments = []; start = 0
    while start < len(labels):
        cls = labels[start]; end = start
        while end+1 < len(labels) and labels[end+1] == cls: end += 1
        mean_mse = float(recon_errors[start:end+1].mean())
        is_anom  = IDX_TO_CLASS[int(cls)] in ANOMALY_CLASSES
        segments.append({
            "class":      IDX_TO_CLASS[int(cls)],
            "start_clip": int(start), "end_clip": int(end),
            "start_s":    int(start*CLIP_LEN), "end_s": int((end+1)*CLIP_LEN),
            "duration_s": int((end-start+1)*CLIP_LEN),
            "mean_mse":   round(mean_mse, 6), "is_anomaly": is_anom,
            "confirmed":  is_anom and mean_mse > threshold,
        }); start = end+1
    return segments

def print_timeline(segments):
    print("\n" + "="*60 + "\nSURVEILLANCE EVENT TIMELINE\n" + "="*60)
    def _fmt(s): m,sc=divmod(s,60); return f"{m:02d}:{sc:02d}"
    for seg in segments:
        flag = "  [ANOMALY CONFIRMED]" if seg["confirmed"] else ""
        print(f"  {_fmt(seg['start_s'])} - {_fmt(seg['end_s'])}  |  "
              f"{seg['class']:<12}  MSE={seg['mean_mse']:.5f}{flag}")
    print("="*60)

def plot_timeline(segments, save_path):
    fig,ax = plt.subplots(figsize=(14,3.5))
    for seg in segments:
        color = TL_COLORS.get(seg["class"],"#AAAAAA")
        ax.barh(0, left=seg["start_s"], width=seg["duration_s"],
                height=0.5, color=color, edgecolor="black", linewidth=0.6)
        if seg["duration_s"] > 30:
            lbl = seg["class"] + (" [!]" if seg["confirmed"] else "")
            ax.text(seg["start_s"]+seg["duration_s"]/2, 0, lbl,
                    ha="center", va="center", fontsize=8, fontweight="bold",
                    color="white" if seg["class"]!="Normal" else "black")
    patches=[mpatches.Patch(color=v,label=k,edgecolor="black",linewidth=0.5)
             for k,v in TL_COLORS.items() if any(s["class"]==k for s in segments)]
    ax.legend(handles=patches,loc="upper right",fontsize=8,ncol=3)
    ax.set_xlabel("Time (seconds)"); ax.set_yticks([])
    ax.set_title("Surveillance Event Timeline",fontsize=13,fontweight="bold")
    ax.set_xlim(0, segments[-1]["end_s"] if segments else 100); ax.grid(axis="x",alpha=0.3)
    for seg in segments:
        if seg["confirmed"]: ax.axvspan(seg["start_s"],seg["end_s"],alpha=0.15,color="red",zorder=0)
    plt.tight_layout(); fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig)
    print(f"[OK] {save_path}")

def plot_score_over_time(recon_errors, pred_labels, threshold, save_path):
    fig,(ax1,ax2)=plt.subplots(2,1,figsize=(14,5),sharex=True)
    t=np.arange(len(recon_errors))
    ax1.plot(t,recon_errors,lw=0.8,color="#333")
    ax1.axhline(threshold,color="red",ls="--",lw=1.2,label=f"Thr={threshold:.4f}")
    ax1.fill_between(t,recon_errors,threshold,where=recon_errors>threshold,alpha=0.3,color="red")
    ax1.set_ylabel("Recon MSE"); ax1.set_title("Anomaly Score Over Time",fontweight="bold")
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
    for i,cls in enumerate(CLASSES):
        mask=pred_labels==i
        ax2.scatter(t[mask],np.full(mask.sum(),i),s=4,color=TL_COLORS.get(cls,"#888"),label=cls,alpha=0.8)
    ax2.set_yticks(range(len(CLASSES))); ax2.set_yticklabels(CLASSES,fontsize=8)
    ax2.set_xlabel("Clip Index"); ax2.set_title("Predicted Class Over Time",fontweight="bold")
    ax2.grid(alpha=0.3)
    plt.tight_layout(); fig.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close(fig)
    print(f"[OK] {save_path}")

print("Timeline helpers ready.")

## 9  Grad-CAM

In [ ]:
class GradCAM:
    """
    Grad-CAM on encoder.cnn[6] (last Conv2d, 64->128 channels).
    Expects clip_np: (T, H, W, 6) float32.
    """
    def __init__(self, encoder, head, device=DEVICE):
        self.encoder = encoder.to(device).eval()
        self.head    = head.to(device)
        self.device  = device
        self._acts = self._grads = None
        target = encoder.cnn[6]  # last Conv2d
        self._fwd_h = target.register_forward_hook(
            lambda m,i,o: setattr(self, "_acts", o)
        )
        self._bwd_h = target.register_full_backward_hook(
            lambda m,gi,go: setattr(self, "_grads", go[0])
        )

    def compute(self, clip_np, class_idx=None, frame_idx=None):
        """clip_np: (T, H, W, 6) float32"""
        if frame_idx is None: frame_idx = CLIP_LEN // 2
        clip_t = (torch.from_numpy(clip_np.transpose(0,3,1,2))
                  .float().unsqueeze(0).to(self.device))  # (1,T,6,H,W)
        # cuDNN LSTM backward requires train mode; encoder stays eval
        self.encoder.eval()
        self.head.train()
        self.encoder.zero_grad(); self.head.zero_grad()
        logits = self.head(self.encoder(clip_t))
        if class_idx is None: class_idx = int(logits.argmax(1).item())
        logits[0, class_idx].backward()
        acts  = self._acts[frame_idx].detach()
        grads = self._grads[frame_idx].detach()
        weights = grads.mean(dim=(1,2))
        cam = F.relu((weights[:,None,None]*acts).sum(0)).cpu().numpy()
        if cam.max() > 0: cam /= cam.max()
        heatmap = np.array(
            Image.fromarray((cam*255).astype(np.uint8))
                 .resize((FRAME_W, FRAME_H), Image.BILINEAR)
        ) / 255.0
        # Use argmax on detached logits for reported class/confidence
        with torch.no_grad():
            probs = torch.softmax(logits.detach(), dim=1)[0]
        return heatmap, IDX_TO_CLASS[class_idx], float(probs[class_idx].item())

    def remove(self):
        self._fwd_h.remove(); self._bwd_h.remove()


def compute_gradcam(encoder, head, clip_np, class_idx=None, frame_idx=None, device=DEVICE):
    gcam   = GradCAM(encoder, head, device)
    result = gcam.compute(clip_np, class_idx, frame_idx)
    gcam.remove(); return result

def overlay_heatmap(frame, heatmap, alpha=0.5):
    heatmap_rgb = cm.get_cmap("jet")(heatmap)[:,:,:3]
    return np.clip((1-alpha)*frame + alpha*heatmap_rgb, 0, 1)

def visualize_gradcam_batch(encoder, head, test_paths, y_test,
                             n_per_class=2, save_path=None, device=DEVICE):
    gcam   = GradCAM(encoder, head, device)
    n_rows = len(ANOMALY_CLASSES) * n_per_class
    fig, axes = plt.subplots(n_rows, 3, figsize=(9, n_rows*2.2))
    if n_rows == 1: axes = axes[np.newaxis,:]
    row = 0
    for cls in ANOMALY_CLASSES:
        cls_idx = CLASS_TO_IDX[cls]
        pool    = np.where(y_test == cls_idx)[0]
        if len(pool) == 0: continue
        for s_idx in np.random.choice(pool, min(n_per_class, len(pool)), replace=False):
            clip_f  = _rgb_to_6ch(_load_frames(test_paths[s_idx]))  # (T,H,W,6)
            mid     = CLIP_LEN // 2
            frame   = clip_f[mid, :, :, :3]
            heatmap, pred_cls, conf = gcam.compute(clip_f, class_idx=cls_idx, frame_idx=mid)
            overlay = overlay_heatmap(frame, heatmap)
            axes[row][0].imshow(frame);              axes[row][0].axis("off")
            axes[row][0].set_title(f"True:{cls}",fontsize=8)
            axes[row][1].imshow(heatmap,cmap="jet"); axes[row][1].axis("off")
            axes[row][1].set_title("Grad-CAM",fontsize=8)
            axes[row][2].imshow(overlay);            axes[row][2].axis("off")
            axes[row][2].set_title(f"Pred:{pred_cls}({conf:.2f})",fontsize=8)
            row += 1
    gcam.remove()
    plt.suptitle("Grad-CAM: Spatial Attention per Anomaly Class",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    if save_path: fig.savefig(save_path,dpi=150,bbox_inches="tight"); print(f"[OK] {save_path}")
    plt.show(); plt.close(fig)

print("Grad-CAM helpers ready.")

---
## ━━━━━━━━━━  P I P E L I N E  ━━━━━━━━━━

## Step 1 — Discover & Split Data

In [ ]:
print("Discovering videos...")
video_dict_train = discover_videos(DATASET_ROOT, "Train")
video_dict_test  = discover_videos(DATASET_ROOT, "Test")

(
    train_paths, y_train,
    val_paths,   y_val,
    test_paths,  y_test,
    normal_train_vids,
) = make_video_splits(video_dict_train, video_dict_test)

normal_paths = build_clips(normal_train_vids, TRAIN_STRIDE, MAX_FRAMES)
print(f"\nNormal pre-train clips: {len(normal_paths):,}")

In [ ]:
class_weights = compute_class_weight(
    "balanced", classes=np.arange(NUM_CLASSES), y=y_train
).astype(np.float32)
print(f"Class weights: {dict(zip(CLASSES, class_weights.round(3)))}")

val_normal_paths = [val_paths[i] for i in range(len(val_paths)) if y_val[i] == CLASS_TO_IDX["Normal"]]
val_anom_paths   = [val_paths[i] for i in range(len(val_paths)) if y_val[i] != CLASS_TO_IDX["Normal"]]
print(f"Val Normal: {len(val_normal_paths)}  |  Val Anomaly: {len(val_anom_paths)}")

print("\nTest set breakdown:")
for cls_name in CLASSES:
    n = (y_test == CLASS_TO_IDX[cls_name]).sum()
    print(f"  {cls_name:<12}: {n}")

## Phase 1 — Autoencoder Pre-training
Trains encoder + decoder on Normal clips only (MSE reconstruction loss).

In [ ]:
encoder     = Encoder()
decoder     = Decoder()
autoencoder = Autoencoder(encoder, decoder)
total = sum(p.numel() for p in autoencoder.parameters())
print(f"Autoencoder params: {total:,}  ({total/1e6:.2f} M)")

In [ ]:
pretrain_history = pretrain_autoencoder(autoencoder, normal_paths, val_split=0.15, device=DEVICE)

In [ ]:
plot_loss(pretrain_history, "Autoencoder Pre-training Loss", PLOTS_DIR/"pretrain_loss.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"pretrain_loss.png")); plt.axis("off"); plt.show()

In [ ]:
# Load 4 random Normal clips from disk for visualisation
sample_np = _load_sample_clips_float(val_normal_paths, n=4)
plot_reconstruction_samples(
    autoencoder, sample_np,
    save_path=PLOTS_DIR/"reconstructions_normal.png", device=DEVICE
)

In [ ]:
plot_mse_distribution(
    autoencoder, val_normal_paths, val_anom_paths,
    save_path=PLOTS_DIR/"mse_distribution.png", device=DEVICE
)
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"mse_distribution.png")); plt.axis("off"); plt.show()

normal_errors = compute_reconstruction_errors(autoencoder, val_normal_paths, device=DEVICE)
thr_2std = compute_threshold(normal_errors, 2.0)
thr_3std = compute_threshold(normal_errors, 3.0)
print(f"Normal MSE  mean={normal_errors.mean():.5f}  std={normal_errors.std():.5f}")
print(f"Threshold 2std = {thr_2std:.5f}")
print(f"Threshold 3std = {thr_3std:.5f}")
_save_json({"mean_2std": thr_2std, "mean_3std": thr_3std}, "thresholds.json")

## Phase 2 — Joint Training (Encoder frozen + LSTM Head)
MSE reconstruction + λ·CrossEntropy. Only the LSTM head parameters are trained.

In [ ]:
encoder = Encoder(); decoder = Decoder()
encoder.load_state_dict(torch.load(MODELS_DIR/"encoder_pretrained.pth", map_location=DEVICE))
decoder.load_state_dict(torch.load(MODELS_DIR/"decoder_pretrained.pth", map_location=DEVICE))
freeze_decoder(decoder)

lstm_head   = LSTMHead()
joint_model = JointModel(encoder, decoder, lstm_head)
print(f"LSTMHead params: {sum(p.numel() for p in lstm_head.parameters()):,}")

In [ ]:
joint_history = joint_train(
    joint_model, train_paths, y_train, val_paths, y_val,
    lam=LAMBDA, device=DEVICE, class_weights=None,
)
print(f"\nBest val acc : {max(joint_history['val_acc']):.4f}")
print(f"Best val loss: {min(joint_history['val_loss']):.4f}")

In [ ]:
# ─── Phase 2b: Train Transformer head and compare with LSTM ──────────────────
print("=" * 60)
print("Phase 2b — Training Attention (Transformer) Head")
print("=" * 60)
enc_attn  = Encoder(); dec_attn = Decoder()
enc_attn.load_state_dict(torch.load(MODELS_DIR/"encoder_pretrained.pth", map_location=DEVICE))
dec_attn.load_state_dict(torch.load(MODELS_DIR/"decoder_pretrained.pth", map_location=DEVICE))
freeze_decoder(dec_attn)
attn_head  = TransformerHead()
joint_attn = JointModel(enc_attn, dec_attn, attn_head)
print(f"Attn head params : {sum(p.numel() for p in attn_head.parameters()):,}")

attn_history = joint_train(
    joint_attn, train_paths, y_train, val_paths, y_val,
    lam=LAMBDA, device=DEVICE, class_weights=None,
    save_path=MODELS_DIR / "joint_attn_best.pth",
)

lstm_best_acc = max(joint_history["val_acc"])
attn_best_acc = max(attn_history["val_acc"])
print(f"\nLSTM head best val acc : {lstm_best_acc:.4f}")
print(f"Attn head best val acc : {attn_best_acc:.4f}")
USE_ATTN = attn_best_acc > lstm_best_acc
winner   = "Attention (Transformer)" if USE_ATTN else "LSTM"
delta    = (attn_best_acc - lstm_best_acc) * 100
print(f"→ Phase 3 will use: {winner}  (Δ = {delta:+.1f}%)")

In [ ]:
plot_joint_loss(joint_history, PLOTS_DIR/"joint_loss.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"joint_loss.png")); plt.axis("off"); plt.show()

# joint_accuracy.png saved separately for report compatibility
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(joint_history["train_acc"], label="Train Accuracy")
ax.plot(joint_history["val_acc"],   label="Val Accuracy", ls="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Classification Accuracy", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(PLOTS_DIR/"joint_accuracy.png", dpi=150, bbox_inches="tight"); plt.show()

## Phase 3 — Encoder Unfreeze Fine-tuning

Loads the best joint checkpoint, unlocks the encoder, and trains end-to-end with differential learning rates:
- **Encoder**: `lr=1e-5` — tiny steps to avoid destroying pretrained representations
- **LSTM Head**: `lr=1e-4` — faster adaptation to updated encoder features
- Cosine annealing schedule, 15 epochs, early stop on val loss (patience 5)
- Same WeightedRandomSampler + label_smoothing=0.1 as Phase 2

In [ ]:
# Load Phase 2 best checkpoint — auto-selects LSTM vs Attention head
if USE_ATTN:
    phase2_ckpt = MODELS_DIR / "joint_attn_best.pth"
    head_ft     = TransformerHead()
    print("Phase 3: using Attention (Transformer) head")
else:
    phase2_ckpt = MODELS_DIR / "joint_model_best.pth"
    head_ft     = LSTMHead()
    print("Phase 3: using LSTM head")

encoder_ft = Encoder(); decoder_ft = Decoder()
encoder_ft.load_state_dict(torch.load(MODELS_DIR/"encoder_pretrained.pth", map_location=DEVICE))
decoder_ft.load_state_dict(torch.load(MODELS_DIR/"decoder_pretrained.pth", map_location=DEVICE))
freeze_decoder(decoder_ft)
joint_ft = JointModel(encoder_ft, decoder_ft, head_ft)
joint_ft.load_state_dict(torch.load(phase2_ckpt, map_location=DEVICE))
print("Loaded Phase 2 best weights — starting encoder fine-tuning...")

finetune_history = finetune_encoder(
    joint_ft, train_paths, y_train, val_paths, y_val,
    lam=FINETUNE_LAMBDA, device=DEVICE, class_weights=None,
)
print(f"\nBest fine-tune val macro F1 : {max(finetune_history['val_macro_f1']):.4f}")

In [ ]:
plot_loss(finetune_history, "Phase 3 — Encoder Fine-tuning Loss", PLOTS_DIR/"finetune_loss.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"finetune_loss.png")); plt.axis("off"); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(finetune_history["train_acc"], label="Train Accuracy")
ax.plot(finetune_history["val_acc"],   label="Val Accuracy", ls="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.set_title("Phase 3: Fine-tune Accuracy", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(PLOTS_DIR/"finetune_accuracy.png", dpi=150, bbox_inches="tight"); plt.show()

# Use finetuned model for all downstream evaluation
joint_model = joint_ft.eval()
encoder     = joint_ft.encoder
lstm_head   = joint_ft.temporal_head
print("Using fine-tuned model for evaluation.")

## Evaluation

In [ ]:
# joint_ft already holds best Phase 3 weights (loaded at end of finetune_encoder)
joint_ft = joint_ft.to(DEVICE).eval()

results, normal_errors, test_errors, binary_gt = threshold_comparison(
    joint_ft, val_normal_paths, test_paths, y_test, device=DEVICE
)
thr_optimal = find_optimal_threshold(test_errors, binary_gt)
print(f"ROC-optimal threshold: {thr_optimal:.5f}")

plot_roc_curve(test_errors, binary_gt, PLOTS_DIR/"roc_curve.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"roc_curve.png")); plt.axis("off"); plt.show()

In [ ]:
plot_anomaly_scores(test_errors, y_test, threshold=thr_optimal,
                   save_path=PLOTS_DIR/"anomaly_scores.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"anomaly_scores.png")); plt.axis("off"); plt.show()

y_pred, report = evaluate_classifier(joint_ft, test_paths, y_test, device=DEVICE)
plot_per_class_metrics(report, PLOTS_DIR/"per_class_metrics.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"per_class_metrics.png")); plt.axis("off"); plt.show()
plot_confusion_matrix(y_test, y_pred, PLOTS_DIR/"confusion_matrix.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"confusion_matrix.png")); plt.axis("off"); plt.show()

In [ ]:
binary_result, p_anomaly, binary_gt_cls = evaluate_binary(
    joint_ft, test_paths, y_test, device=DEVICE
)
plot_binary_roc(p_anomaly, binary_gt_cls, PLOTS_DIR/"binary_roc.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"binary_roc.png")); plt.axis("off"); plt.show()

In [ ]:
# Sample up to 2000 test clips for embedding visualisation (speed)
n_emb    = min(2000, len(test_paths))
emb_paths = test_paths[:n_emb]
emb_labels = y_test[:n_emb]
embeddings = get_embeddings(joint_ft.encoder.to(DEVICE), emb_paths, device=DEVICE)
print(f"Embeddings: {embeddings.shape}")
plot_latent_space(embeddings, emb_labels, method="pca",  save_path=PLOTS_DIR/"pca_latent.png")
plot_latent_space(embeddings, emb_labels, method="tsne", save_path=PLOTS_DIR/"tsne_latent.png")

## Timeline

In [ ]:
seq_paths, seq_labels, segment_info = build_test_sequence(
    test_paths, y_test, clips_per_segment=20
)
print(f"Sequence: {len(seq_paths)} clips")
for cls, s, e in segment_info:
    print(f"  [{cls}] clips {s}–{e}")

pred_labels, pred_probs, recon_errors_tl = sliding_window_predict(
    joint_ft, seq_paths, device=DEVICE
)
smoothed = majority_vote_smooth(pred_labels, window=3)
segments = merge_segments(smoothed, recon_errors_tl, threshold=thr_optimal)
print_timeline(segments)

In [ ]:
plot_score_over_time(recon_errors_tl, pred_labels, thr_optimal, PLOTS_DIR/"score_over_time.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"score_over_time.png")); plt.axis("off"); plt.show()

plot_timeline(segments, PLOTS_DIR/"timeline.png")
plt.figure(); plt.imshow(plt.imread(PLOTS_DIR/"timeline.png")); plt.axis("off"); plt.show()

## Grad-CAM

In [ ]:
shop_idx  = np.where(y_test == CLASS_TO_IDX["Shoplifting"])[0]
clip_f    = _rgb_to_6ch(_load_frames(test_paths[fight_idx[0]]))  # (T, H, W, 6)

heatmap, pred_cls, conf = compute_gradcam(
    joint_ft.encoder.to(DEVICE), joint_ft.temporal_head.to(DEVICE), clip_f,
    class_idx=CLASS_TO_IDX["Shoplifting"], frame_idx=CLIP_LEN//2, device=DEVICE
)
frame   = clip_f[CLIP_LEN//2, :, :, :3]
overlay = overlay_heatmap(frame, heatmap, alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(frame);               axes[0].set_title("Input Frame");  axes[0].axis("off")
axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Grad-CAM");     axes[1].axis("off")
axes[2].imshow(overlay);             axes[2].set_title(f"Pred: {pred_cls} ({conf:.2f})"); axes[2].axis("off")
plt.suptitle("Grad-CAM — Shoplifting", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR/"gradcam_example.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
visualize_gradcam_batch(
    joint_ft.encoder.to(DEVICE), joint_ft.temporal_head.to(DEVICE),
    test_paths, y_test, n_per_class=2,
    save_path=PLOTS_DIR/"gradcam_batch.png", device=DEVICE
)

## Ablation Study
- **A** Single-frame CNN (no temporal modelling)  
- **B** CNN + TemporalPool (mean-pool head)  
- **C** CNN + LSTM frozen decoder *(ours)*

In [ ]:
print("="*50 + "\nVARIANT A: Single-frame CNN\n" + "="*50)

sf_cnn = SingleFrameCNN().to(DEVICE)
sf_opt = torch.optim.Adam(sf_cnn.parameters(), lr=LEARNING_RATE)
wt_t   = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
ce_fn  = nn.CrossEntropyLoss(weight=wt_t)

tr_sf = DataLoader(
    MidFrameDataset(train_paths, y_train, training=True),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0)
)
va_sf = DataLoader(
    MidFrameDataset(val_paths, y_val, training=False),
    batch_size=EVAL_BS, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0)
)

best_sf = float("inf"); pat = 0
for epoch in range(JOINT_EPOCHS):
    sf_cnn.train()
    for x, y in tr_sf:
        x, y = x.to(DEVICE), y.to(DEVICE)
        sf_opt.zero_grad(); ce_fn(sf_cnn(x), y).backward(); sf_opt.step()
    sf_cnn.eval(); vl = 0.0
    with torch.no_grad():
        for x, y in va_sf: vl += ce_fn(sf_cnn(x.to(DEVICE)), y.to(DEVICE)).item()
    if vl < best_sf:
        best_sf = vl; pat = 0
        torch.save(sf_cnn.state_dict(), MODELS_DIR/"ablation_sf.pth")
    else:
        pat += 1
        if pat >= 5: print(f"Early stop @ {epoch+1}"); break

sf_cnn.load_state_dict(torch.load(MODELS_DIR/"ablation_sf.pth", map_location=DEVICE))
sf_wrapper = SingleFrameWrapper(sf_cnn).to(DEVICE).eval()
_, report_a = evaluate_classifier(sf_wrapper, test_paths, y_test, device=DEVICE)
sf_f1 = report_a["macro avg"]["f1-score"]
print(f"Variant A macro F1: {sf_f1:.4f}")

In [ ]:
print("="*50 + "\nVARIANT B: CNN + TemporalPool\n" + "="*50)
enc_b = Encoder(); dec_b = Decoder(); head_b = TemporalPoolHead()
enc_b.load_state_dict(torch.load(MODELS_DIR/"encoder_pretrained.pth", map_location=DEVICE))
dec_b.load_state_dict(torch.load(MODELS_DIR/"decoder_pretrained.pth", map_location=DEVICE))
freeze_decoder(dec_b)
joint_b = JointModel(enc_b, dec_b, head_b)
hist_b  = joint_train(
    joint_b, train_paths, y_train, val_paths, y_val,
    device=DEVICE, class_weights=None,
    save_path=MODELS_DIR/"ablation_pool_best.pth",
)
_, report_b = evaluate_classifier(joint_b.to(DEVICE), test_paths, y_test, device=DEVICE)
b_f1     = report_b["macro avg"]["f1-score"]
b_errors = compute_reconstruction_errors(joint_b, test_paths, device=DEVICE)
b_auc    = roc_auc_score((y_test!=CLASS_TO_IDX["Normal"]).astype(int), b_errors)
print(f"Variant B macro F1: {b_f1:.4f}")

print("="*50 + "\nVARIANT C: CNN + LSTM frozen dec. (ours)\n" + "="*50)
_, report_c = evaluate_classifier(joint_model, test_paths, y_test, device=DEVICE)
c_f1     = report_c["macro avg"]["f1-score"]
c_errors = compute_reconstruction_errors(joint_model, test_paths, device=DEVICE)
c_auc    = roc_auc_score((y_test!=CLASS_TO_IDX["Normal"]).astype(int), c_errors)
print(f"Variant C macro F1: {c_f1:.4f}")

print("="*50 + "\nVARIANT D: CNN + Attention (Transformer)\n" + "="*50)
joint_attn.eval()
_, report_d = evaluate_classifier(joint_attn.to(DEVICE), test_paths, y_test, device=DEVICE)
d_f1     = report_d["macro avg"]["f1-score"]
d_errors = compute_reconstruction_errors(joint_attn, test_paths, device=DEVICE)
d_auc    = roc_auc_score((y_test!=CLASS_TO_IDX["Normal"]).astype(int), d_errors)
print(f"Variant D macro F1: {d_f1:.4f}")

print("\n" + "="*70 + "\nABLATION SUMMARY\n" + "="*70)
print(f"{'Model':<45} {'AUC':>8} {'Macro F1':>10}"); print("-"*70)
print(f"  A: Single-frame CNN                          {'N/A':>8} {sf_f1:>10.4f}")
print(f"  B: CNN + TemporalPool                   {b_auc:>8.4f} {b_f1:>10.4f}")
print(f"  C: CNN + LSTM, frozen dec. (Phase2)     {c_auc:>8.4f} {c_f1:>10.4f}")
print(f"  D: CNN + Attention, frozen dec.         {d_auc:>8.4f} {d_f1:>10.4f}")
print("="*70)

variants = ["A: Single-frame\nCNN", "B: CNN+\nTemporalPool",
            "C: CNN+LSTM\nFrozen", "D: CNN+\nAttention"]
scores   = [sf_f1, b_f1, c_f1, d_f1]
colors   = ["#AAAAAA", "#888888", "#1E88E5", "#43A047"]
fig, ax  = plt.subplots(figsize=(9, 4))
bars = ax.bar(variants, scores, color=colors, edgecolor="black", lw=0.8)
ax.set_ylim(0, 1.1); ax.set_ylabel("Macro F1")
ax.set_title("Ablation: Macro F1 by Variant", fontweight="bold")
for bar, sc in zip(bars, scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f"{sc:.4f}", ha="center", fontsize=10, fontweight="bold")
ax.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.savefig(PLOTS_DIR/"ablation_f1.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"[OK] {PLOTS_DIR/'ablation_f1.png'}")

best_name = ["A","B","C (LSTM)","D (Attention)"][scores.index(max(scores))]
print(f"\nBest variant by macro F1: {best_name}")

## Done

All outputs saved to `./outputs/`.

| Folder | Contents |
|---|---|
| `outputs/models/` | `autoencoder_best.pth`, `joint_model_best.pth`, ablation checkpoints |
| `outputs/plots/`  | Loss curves, confusion matrix, ROC, Grad-CAM, timeline, latent space |
| `outputs/results/`| JSON metrics: classification report, binary report, thresholds |